In [1]:
# DIVERGES FROM SUBSTRATE cell 0 — sys.path adjusted for notebooks/ location (one parent dir up vs replication/notebooks/).
import os, sys
_HERE = os.path.abspath(os.getcwd())
for p in (os.path.abspath(os.path.join(_HERE, "..", "reference")),
          os.path.abspath(os.path.join(_HERE, "..", "replication"))):
    if p not in sys.path:
        sys.path.insert(0, p)

# networkx 3.x removed from_numpy_matrix; reference/stats_count.py still calls it.
# Behavior of from_numpy_array on a 2D ndarray matches the old from_numpy_matrix.
import networkx as _nx
if not hasattr(_nx, "from_numpy_matrix"):
    _nx.from_numpy_matrix = _nx.from_numpy_array


In [2]:
# DIVERGES FROM SUBSTRATE cell 1 — drop BertForSequenceClassification (unused); add BertTokenizerFast; keep grab_attention_weights, text_preprocessing (ripser path, not embedding extraction).
from collections import defaultdict
import itertools
import re
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from transformers import BertTokenizerFast, BertModel

from stats_count import *
from grab_weights import grab_attention_weights, text_preprocessing


In [3]:
import warnings

warnings.filterwarnings('ignore')

In [4]:
!env | grep CUDA_VISIBLE

## Parameters

In [5]:
np.random.seed(42) # For reproducibility.

In [6]:
# DIVERGES FROM SUBSTRATE cell 6 — max_tokens 128→32 for KWIC contexts; model bert-base-uncased→bert-base-multilingual-cased.
max_tokens_amount  = 32  # KWIC contexts are ~21 whitespace tokens; 32 leaves headroom
                         # for [CLS]/[SEP] and ru/es subword expansion.

layers_of_interest = [i for i in range(12)]  # Layers for which attention matrices and features on them are 
                                             # calculated. For calculating features on all layers, leave it be
                                             # [i for i in range(12)].

model_path = tokenizer_path = "bert-base-multilingual-cased"

# You can use either standard or fine-tuned BERT. If you want to use fine-tuned BERT to your current task, save the
# model and the tokenizer with the commands tokenizer.save_pretrained(output_dir); 
# bert_classifier.save_pretrained(output_dir) into the same directory and insert the path to it here.


### Explanation of stats_name parameter

Currently, we implemented calculation of the following graphs features:
* "s"    - amount of strongly connected components
* "w"    - amount of weakly connected components
* "e"    - amount of edges
* "v"    - average vertex degree
* "c"    - amount of (directed) simple cycles
* "b0b1" - Betti numbers

The variable stats_name contains a string with the names of the features, which you want to calculate. The format of the string is the following:

"stat_name + "_" + stat_name + "_" + stat_name + ..."

**For example**:

`stats_name == "s_w"` means that the number of strongly and weakly connected components will be calculated

`stats_name == "b0b1"` means that only the Betti numbers will be calculated

`stats_name == "b0b1_c"` means that Betti numbers and the number of simple cycles will be calculated

e.t.c.

## Filenames

In [7]:
# DIVERGES FROM SUBSTRATE cell 9 — subset variable replaced with hardcoded lang/domain; paths repointed to
# data/kwic/, data/phase3/attentions/, data/phase3/barcodes/, results/phase3_ripser/, results/phase3_templates/.
# ---------------------------------------------------------------------------
# SCOPE — COSI 115a final analysis is COLOR-only (rescoped 2026-05-04).
# Emotion and kinship CSVs exist under data/kwic/ but are out of scope.
# See `bd show ph-project-mwk` and CLAUDE.md "Current scope" for rationale.
#
# To process all (lang, color) pairs, edit `lang` below and re-run the
# notebook from this cell downward (kernel restart between passes recommended).
# ---------------------------------------------------------------------------
lang   = "en"        # one of: "en", "ru", "es"
domain = "color"     # COSI 115a scope: color only

input_dir    = "../data/kwic/"
attn_dir     = "../data/phase3/attentions/"
barcode_dir  = "../data/phase3/barcodes/"
ripser_dir   = "../results/phase3_ripser/"
template_dir = "../results/phase3_templates/"
os.makedirs(barcode_dir, exist_ok=True)
os.makedirs(ripser_dir, exist_ok=True)
os.makedirs(template_dir, exist_ok=True)

prefix = ripser_dir + lang + "_" + domain

r_file = attn_dir + lang + "_" + domain + "_all_heads_" + str(len(layers_of_interest)) + "_layers_MAX_LEN_" + \
         str(max_tokens_amount) + "_" + model_path.split("/")[-1]
# Name of the file for attention matrices weights — must match r_file in mbert_attention_thresholds.ipynb cell 10
# so that substring matching in cell 24 resolves the correct part*.npy files.

barcodes_file = barcode_dir + lang + "_" + domain + "_all_heads_" + str(len(layers_of_interest)) + "_layers_MAX_LEN_" + \
                str(max_tokens_amount) + "_" + model_path.split("/")[-1]
# Name of the file for barcodes information


In [8]:
r_file

'../data/phase3/attentions/en_color_all_heads_12_layers_MAX_LEN_32_bert-base-multilingual-cased'

In [9]:
barcodes_file

'../data/phase3/barcodes/en_color_all_heads_12_layers_MAX_LEN_32_bert-base-multilingual-cased'

.csv file must contain the column with the name **sentence** with the texts. It can also contain the column **labels**, which will be needed for testing. Any other arbitrary columns will be ignored.

In [10]:
# DIVERGES FROM SUBSTRATE cell 13 — input path subset+'.csv' → lang+'/'+domain+'.csv'.
try:
    data = pd.read_csv(input_dir + lang + "/" + domain + ".csv").reset_index(drop=True)
except:
    #data = pd.read_csv(input_dir + lang + "/" + domain + ".tsv", delimiter="\t")
    data = pd.read_csv(input_dir + lang + "/" + domain + ".tsv", delimiter="\t", header=None)
    data.columns = ["0", "labels", "2", "sentence"]


In [11]:
data.head()

,term,labels,sentence,target_idx,corpus_source
0,black,black,health care system and from providers often mi...,10,eng_news_2023_1M
1,black,black,The critically endangered black rhino populati...,3,eng_news_2020_1M
2,black,black,adopts sci-fi to address alienation and aspira...,10,eng_news_2019_1M
3,black,black,A governmental financial black hole into which...,3,eng_news_2019_1M
4,black,black,When my Black female friend fell into a migrai...,2,eng_news_2019_1M


In [12]:
sentences = data['sentence']
print("Average amount of words in example:", \
      np.mean(list(map(len, map(lambda x: re.sub('\w', ' ', x).split(" "), data['sentence'])))))
print("Max. amount of words in example:", \
      np.max(list(map(len, map(lambda x: re.sub('\w', ' ', x).split(" "), data['sentence'])))))
print("Min. amount of words in example:", \
      np.min(list(map(len, map(lambda x: re.sub('\w', ' ', x).split(" "), data['sentence'])))))

Average amount of words in example: 98.04954545454545
Max. amount of words in example: 160
Min. amount of words in example: 32


In [ ]:
# DIVERGES FROM SUBSTRATE cell 16 — deprecated batch_encode_plus + pad_to_max_length removed (transformers 5.x); replaced with modern tokenizer() call. Same migration as patch_notebook.py applies to the threshold substrate.
def get_token_length(batch_texts):
    inputs = tokenizer(list(batch_texts),
       return_tensors='pt',
       add_special_tokens=True,
       max_length=MAX_LEN,             # Max length to truncate/pad
       padding="max_length",         # Pad sentence to max length
       truncation=True
    )
    inputs = inputs['input_ids'].numpy()
    n_tokens = []
    indexes = np.argwhere(inputs == tokenizer.pad_token_id)
    for i in range(inputs.shape[0]):
        ids = indexes[(indexes == i)[:, 0]]
        if not len(ids):
            n_tokens.append(MAX_LEN)
        else:
            n_tokens.append(ids[0, 1])
    return n_tokens

In [14]:
# DIVERGES FROM SUBSTRATE cell 17 — BertTokenizer→BertTokenizerFast; do_lower_case True→False (Russian/Spanish casing carries meaning).
MAX_LEN = max_tokens_amount
tokenizer = BertTokenizerFast.from_pretrained(tokenizer_path, do_lower_case=False)  # Russian and Spanish casing carries meaning.


In [15]:
data['tokenizer_length'] = get_token_length(data['sentence'].values)

AttributeError: BertTokenizer has no attribute batch_encode_plus

In [ ]:
data

In [ ]:
ntokens_array = data['tokenizer_length'].values

In [ ]:
from math import ceil

batch_size = 10 # batch size
number_of_batches = ceil(len(data['sentence']) / batch_size)
DUMP_SIZE = 100 # number of batches to be dumped

## Calculating Ripser features

Format: "h{dim}\_{type}\_{args}"

Dimension: 0, 1, etc.; homology dimension

Types: 
    
    1. s: sum of lengths; example: "h1_s".
    2. m: mean of lengths; example: "h1_m"
    3. v: variance of lengths; example "h1_v"
    4. n: number of barcodes with time of birth/death more/less then threshold.
        4.1. b/d: birth or death
        4.2. m/l: more or less than threshold
        4.2. t: threshold value
       example: "h0_n_d_m_t0.5", "h1_n_b_l_t0.75"
    5. t: time of birth/death of the longest barcode (not incl. inf).
        3.1. b/d: birth of death
        example: "h0_t_d", "h1_t_b"
    6. nb: number of barcodes in dim
       example: h0_nb
    7. e: entropy; example: "h1_e"

In [ ]:
# DIVERGES FROM SUBSTRATE cell 24 — output_dir + 'attentions/' → attn_dir.
import os
import timeit
import ripser_count

adj_filenames = [
    attn_dir + filename 
    for filename in os.listdir(attn_dir) if r_file in (attn_dir + filename)
]
# sorted by part number
adj_filenames = sorted(adj_filenames, key = lambda x: int(x.split('_')[-1].split('of')[0][4:].strip())) 
adj_filenames


In [ ]:
dim = 1
lower_bound = 1e-3

## Calculating and saving barcodes

In [ ]:
from multiprocessing import Process, Queue

def subprocess_wrap(queue, function, args):
    queue.put(function(*args))
#     print("Putted in Queue")
    queue.close()
    exit()

In [ ]:
import json
import itertools
from collections import defaultdict

def get_only_barcodes(filename, indices, ntokens_array, dim, lower_bound):
    """Get barcodes from a slice of an attention .npy. Loads via
    mmap so the child process never materializes more than its
    slice; parent never materializes the array body at all."""
    adj_matricies = np.load(filename, mmap_mode='r')[indices]
    barcodes = {}
    layers, heads = range(adj_matricies.shape[1]), range(adj_matricies.shape[2])
    for (layer, head) in itertools.product(layers, heads):
        matricies = adj_matricies[:, layer, head, :, :]
        barcodes[(layer, head)] = ripser_count.get_barcodes(
            matricies, ntokens_array, dim, lower_bound, (layer, head))
    return barcodes

def format_barcodes(barcodes):
    """Reformat barcodes to json-compatible format"""
    return [{d: b[d].tolist() for d in b} for b in barcodes]

def save_barcodes(barcodes, filename):
    """Save barcodes to file"""
    formatted_barcodes = defaultdict(dict)
    for layer, head in barcodes:
        formatted_barcodes[layer][head] = format_barcodes(barcodes[(layer, head)])
    json.dump(formatted_barcodes, open(filename, 'w'))
    
def unite_barcodes(barcodes, barcodes_part):
    """Unite 2 barcodes"""
    for (layer, head) in barcodes_part:
        barcodes[(layer, head)].extend(barcodes_part[(layer, head)])
    return barcodes

def split_matricies_and_lengths(adj_matricies, ntokens, number_of_splits):
    splitted_ids = np.array_split(np.arange(ntokens.shape[0]), number_of_splits) 
    splitted = [(adj_matricies[ids], ntokens[ids]) for ids in splitted_ids]
    return splitted

In [ ]:
queue = Queue()
number_of_splits = 20
for i, filename in enumerate(tqdm(adj_filenames, desc='Calculating barcodes')):
    barcodes = defaultdict(list)
    print(f"Processing: {filename}")
    ntokens = ntokens_array[i*batch_size*DUMP_SIZE : (i+1)*batch_size*DUMP_SIZE]
    splitted_ids = np.array_split(np.arange(len(ntokens)), number_of_splits)
    for indices in tqdm(splitted_ids, leave=False):
        p = Process(
            target=subprocess_wrap,
            args=(
                queue,
                get_only_barcodes,
                (filename, indices, ntokens[indices], dim, lower_bound)
            )
        )
        p.start()
        barcodes_part = queue.get()
        p.join()
        p.close()
        barcodes = unite_barcodes(barcodes, barcodes_part)
    part = filename.split('_')[-1].split('.')[0]
    save_barcodes(barcodes, barcodes_file + '_' + part + '.json')

## Calculating features of saved barcodes

In [ ]:
ripser_feature_names=[
    'h0_s', 
    'h0_e',
    'h0_t_d', 
    'h0_n_d_m_t0.75',
    'h0_n_d_m_t0.5',
    'h0_n_d_l_t0.25',
    'h1_t_b',
    'h1_n_b_m_t0.25',
    'h1_n_b_l_t0.95', 
    'h1_n_b_l_t0.70',  
    'h1_s',
    'h1_e',
    'h1_v',
    'h1_nb'
]

In [ ]:
# DIVERGES FROM SUBSTRATE cell 32 — output_dir + 'barcodes/' → barcode_dir.
import os
import timeit
import ripser_count
import json

adj_filenames = [
    barcode_dir + filename 
    for filename in os.listdir(barcode_dir) if r_file.split('/')[-1] == filename.split('_part')[0]
]
adj_filenames = sorted(adj_filenames, key = lambda x: int(x.split('_')[-1].split('of')[0][4:].strip())) 
adj_filenames


In [ ]:
def reformat_barcodes(barcodes):
    """Return barcodes to their original format"""
    formatted_barcodes = []
    for barcode in barcodes:
        formatted_barcode = {}
        for dim in barcode:
            formatted_barcode[int(dim)] = np.asarray(
                [(b, d) for b,d in barcode[dim]], dtype=[('birth', '<f4'), ('death', '<f4')]
            )
        formatted_barcodes.append(formatted_barcode)
    return formatted_barcodes

In [ ]:
features_array = []

for filename in tqdm(adj_filenames, desc='Calculating ripser++ features'):
    barcodes = json.load(open(filename))
    print(f"Barcodes loaded from: {filename}", flush=True)
    features_part = []
    for layer in barcodes:
        features_layer = []
        for head in barcodes[layer]:
            ref_barcodes = reformat_barcodes(barcodes[layer][head])
            features = ripser_count.count_ripser_features(ref_barcodes, ripser_feature_names)
            features_layer.append(features)
        features_part.append(features_layer)
    features_array.append(np.asarray(features_part))

In [ ]:
# DIVERGES FROM SUBSTRATE cell 35 — output path repointed to ripser_dir + lang/domain prefix.
ripser_file = ripser_dir + lang + "_" + domain + "_all_heads_" + str(len(layers_of_interest)) + "_layers" \
             + "_MAX_LEN_" + str(max_tokens_amount) + \
             "_" + model_path.split("/")[-1] + "_ripser" + '.npy'
ripser_file


In [ ]:
features = np.concatenate(features_array, axis=2)
features.shape

In [ ]:
np.save(ripser_file, features)

## Calculating template features

In [ ]:
import os
from multiprocessing import Pool

In [ ]:
# DIVERGES FROM SUBSTRATE cell 40 — BertTokenizer→BertTokenizerFast; bert-base-uncased→tokenizer_path; do_lower_case True→False.
tokenizer = BertTokenizerFast.from_pretrained(tokenizer_path, do_lower_case=False)


In [ ]:
def matrix_distance(matricies, template, broadcast=True):
    """
    Calculates the distance between the list of matricies and the template matrix.
    Args:
    
    -- matricies: np.array of shape (n_matricies, dim, dim)
    -- template: np.array of shape (dim, dim) if broadcast else (n_matricies, dim, dim)
    
    Returns:
    -- diff: np.array of shape (n_matricies, )
    """
    diff = np.linalg.norm(matricies-template, ord='fro', axis=(1, 2))
    div = np.linalg.norm(matricies, ord='fro', axis=(1, 2))**2
    if broadcast:
        div += np.linalg.norm(template, ord='fro')**2
    else:
        div += np.linalg.norm(template, ord='fro', axis=(1, 2))**2
    return diff/np.sqrt(div)

In [ ]:
# DIVERGES FROM SUBSTRATE cell 42 — paths repointed using cell 9 vars (attn_dir, lang, domain, model_path, max_tokens_amount).
attention_dir = attn_dir
attention_name = lang + "_" + domain + "_all_heads_" + str(len(layers_of_interest)) + "_layers_MAX_LEN_" + \
                 str(max_tokens_amount) + "_" + model_path.split("/")[-1]
texts_name = input_dir + lang + "/" + domain + ".csv"
MAX_LEN = max_tokens_amount


In [ ]:
def attention_to_self(matricies):
    """
    Calculates the distance between input matricies and identity matrix, 
    which representes the attention to the same token.
    """
    _, n, m = matricies.shape
    assert n == m, f"Input matrix has shape {n} x {m}, but the square matrix is expected"
    template_matrix = np.eye(n)
    return matrix_distance(matricies, template_matrix)

def attention_to_next_token(matricies):
    """
    Calculates the distance between input and E=(i, i+1) matrix, 
    which representes the attention to the next token.
    """
    _, n, m = matricies.shape
    assert n == m, f"Input matrix has shape {n} x {m}, but the square matrix is expected"
    template_matrix = np.triu(np.tri(n, k=1, dtype=matricies.dtype), k=1)
    return matrix_distance(matricies, template_matrix)

def attention_to_prev_token(matricies):
    """
    Calculates the distance between input and E=(i+1, i) matrix, 
    which representes the attention to the previous token.
    """
    _, n, m = matricies.shape
    assert n == m, f"Input matrix has shape {n} x {m}, but the square matrix is expected"
    template_matrix = np.triu(np.tri(n, k=-1, dtype=matricies.dtype), k=-1)
    return matrix_distance(matricies, template_matrix)

def attention_to_beginning(matricies):
    """
    Calculates the distance between input and E=(i+1, i) matrix, 
    which representes the attention to [CLS] token (beginning).
    """
    _, n, m = matricies.shape
    assert n == m, f"Input matrix has shape {n} x {m}, but the square matrix is expected"
    template_matrix = np.zeros((n, n))
    template_matrix[:, 0] = 1.0
    return matrix_distance(matricies, template_matrix)

def attention_to_ids(matricies, list_of_ids, token_id):
    """
    Calculates the distance between input and ids matrix, 
    which representes the attention to some particular tokens,
    which ids are in the `list_of_ids` (commas, periods, separators).
    """
   
    batch_size, n, m = matricies.shape
    EPS = 1e-7
    assert n == m, f"Input matrix has shape {n} x {m}, but the square matrix is expected"
#     assert len(list_of_ids) == batch_size, f"List of ids length doesn't match the dimension of the matrix"
    template_matrix = np.zeros_like(matricies)
    ids = np.argwhere(list_of_ids == token_id)
    if len(ids):
        batch_ids, row_ids = zip(*ids)
        template_matrix[np.array(batch_ids), :, np.array(row_ids)] = 1.0
        template_matrix /= (np.sum(template_matrix, axis=-1, keepdims=True) + EPS)
    return matrix_distance(matricies, template_matrix, broadcast=False)

In [ ]:
# DIVERGES FROM SUBSTRATE cell 44 — comma_id 1010→117 and dot_id 1012→119 for bert-base-multilingual-cased vocab.
def count_template_features(matricies, feature_list=['self', 'beginning', 'prev', 'next', 'comma', 'dot'], ids=None):
    features = []
    comma_id = 117   # ',' in bert-base-multilingual-cased (was 1010 for bert-base-uncased)
    dot_id   = 119   # '.' in bert-base-multilingual-cased (was 1012 for bert-base-uncased)
    for feature in feature_list:
        if feature == 'self':
            features.append(attention_to_self(matricies))
        elif feature == 'beginning':
            features.append(attention_to_beginning(matricies))
        elif feature == 'prev':
            features.append(attention_to_prev_token(matricies))
        elif feature == 'next':
            features.append(attention_to_next_token(matricies))
        elif feature == 'comma':
            features.append(attention_to_ids(matricies, ids, comma_id))
        elif feature == 'dot':
            features.append(attention_to_ids(matricies, ids, dot_id))
    return np.array(features)

def calculate_features_t(adj_matricies, template_features, ids=None):
    """Calculate template features for adj_matricies"""
    features = []
    for layer in range(adj_matricies.shape[1]):
        features.append([])
        for head in range(adj_matricies.shape[2]):
            matricies = adj_matricies[:, layer, head, :, :]
            lh_features = count_template_features(matricies, template_features, ids) # samples X n_features
            features[-1].append(lh_features)
    return np.asarray(features) # layer X head X n_features X samples


In [ ]:
# DIVERGES FROM SUBSTRATE cell 45 — deprecated batch_encode_plus + pad_to_max_length removed; replaced with modern tokenizer() call.
# '.' id 119  (bert-base-multilingual-cased; was 1012 for bert-base-uncased)
# ',' id 117  (bert-base-multilingual-cased; was 1010 for bert-base-uncased)
def get_list_of_ids(sentences, tokenizer):
    inputs = tokenizer([text_preprocessing(s) for s in sentences],
                       add_special_tokens=True,
                       max_length=MAX_LEN,
                       padding="max_length",
                       truncation=True)
    return np.array(inputs['input_ids'])


In [ ]:
num_of_workers = 20 
pool = Pool(num_of_workers)
feature_list = ['self', 'beginning', 'prev', 'next', 'comma', 'dot']

In [ ]:
adj_filenames = [
    attention_dir + filename 
    for filename in os.listdir(attention_dir) 
    if attention_name == filename.split("_part")[0]
]
# sorted by part number
adj_filenames = sorted(adj_filenames, key = lambda x: int(x.split('_')[-1].split('of')[0][4:].strip())) 
adj_filenames

In [ ]:
attention_name

In [ ]:
texts = pd.read_csv(texts_name)

In [ ]:
features_array = []

for i, filename in tqdm(list(enumerate(adj_filenames)), desc='Features calc'):
    adj_matricies = np.load(filename, allow_pickle=True)
    batch_size = adj_matricies.shape[0]
    sentences = texts['sentence'].values[i*batch_size:(i+1)*batch_size]
    splitted_indexes = np.array_split(np.arange(batch_size), num_of_workers)
    splitted_list_of_ids = [
        get_list_of_ids(sentences[indx], tokenizer) 
        for indx in tqdm(splitted_indexes, desc=f"Calculating token ids on iter {i} from {len(adj_filenames)}")
    ]
    splitted_adj_matricies = [adj_matricies[indx] for indx in splitted_indexes]
    
    args = [(m, feature_list, list_of_ids) for m, list_of_ids in zip(splitted_adj_matricies, splitted_list_of_ids)]
    
    features_array_part = pool.starmap(
        calculate_features_t, args
    )
    features_array.append(np.concatenate([_ for _ in features_array_part], axis=3))
features_array = np.concatenate(features_array, axis=3)

In [ ]:
# DIVERGES FROM SUBSTRATE cell 51 — output path small_gpt_web/features/ → template_dir.
template_dir + attention_name + "_template.npy"


In [ ]:
# DIVERGES FROM SUBSTRATE cell 52 — output path small_gpt_web/features/ → template_dir.
np.save(template_dir + attention_name + "_template.npy", features_array)
